# eq_waves1: equatorial waves in a one-layer model on the equatorial beta plane
pyOM2: py_config/eq_waves1.py

One layer with the equivalent depth c_n^2/g, implicit free surface. The initial
condition is one of the linear equatorial wave solutions, chosen with the variable
wave: 'kelvin', 'yanai', 'rossby' or 'gravity', with meridional mode m and zonal
wavenumber kx = KX pi/Lx.

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PYOM3_ROOT', os.path.abspath('../..')))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.pe_realistic.model import model
from numpy.polynomial.hermite import Hermite

In [ ]:
wave       = 'kelvin'
m          = 1                      # meridional mode (yanai, rossby, gravity)
KX         = 2.0                    # zonal wavenumber in units of pi/Lx
beta       = 2e-11
Re         = 500e3                  # equatorial Rossby radius
cn         = Re**2*beta             # gravity wave speed
A          = 0.01                   # amplitude
nx, ny, nz = 64, 64, 1
dx, dt     = 100e3, 1800.
Lx         = nx*dx
y0         = ny*dx/2                # equator
nsteps     = 1440                   # 30 days
plot_every = 48

par = dict(nx=nx, ny=ny, nz=nz, dxt=dx, dyt=dx, dzt=cn**2/9.81, dt=dt,
           coord_degree=False, set_zero_at_surface=True,
           periodic_in_x=True, periodic_in_y=False,
           eq_of_state=0, implicit_free_surface=True, explicit_free_surface=False,
           epsilon=1e-12, max_iterations=5000, superbee_advection=True)

In [ ]:
def Hf(m, y):                       # Hermite function
    if m < 0: return 0*y
    return np.exp(-0.5*y**2)*Hermite.basis(m)(y)


class eq_waves1(model):

    def set_coriolis(self):
        f = beta*(self.yt - y0)
        self.coriolis_t = OM.np.asarray(f[:, None] + 0*self.xt[None, :])

    def set_initial_conditions(self):
        kx = KX*np.pi/Lx
        XT, YT = np.meshgrid(self.xt_gl, self.yt_gl - y0)       # T points
        XU, YU = np.meshgrid(self.xu_gl, self.yu_gl - y0)       # U and V points
        u = 0*XT; v = 0*XT; p_s = 0*XT
        if wave == 'kelvin':
            u   = A*np.exp(-0.5*YT**2/Re**2)*np.cos(kx*XU)/cn
            p_s = A*np.exp(-0.5*YT**2/Re**2)*np.cos(kx*XT)
        if wave == 'yanai':
            om  = kx*cn/2 + np.sqrt(beta*cn + (kx*cn/2)**2)
            v   = A*np.exp(-0.5*YU**2/Re**2)*np.cos(kx*XT)
            u   = -A*beta*YT/(om - cn*kx)*np.exp(-0.5*YT**2/Re**2)*np.sin(kx*XU)
            p_s = -A*beta*YT/(om - cn*kx)*np.exp(-0.5*YT**2/Re**2)*np.sin(kx*XT)*cn
        if wave in ('gravity', 'rossby'):
            if wave == 'gravity': om = cn*np.sqrt(kx**2 + 2*m/Re**2 + 1/Re**2)
            else:                 om = -beta*kx/(kx**2 + 2*m/Re**2 + 1/Re**2)
            v   = A*Hf(m, YU/Re)*np.cos(kx*XT)
            pp  = Hf(m + 1, YT/Re)/(om - cn*kx) + 2*m*Hf(m - 1, YT/Re)/(om + cn*kx)
            u   = -A*0.5*np.sqrt(cn*beta)*pp*np.sin(kx*XU)
            pp  = Hf(m + 1, YT/Re)/(om - cn*kx) - 2*m*Hf(m - 1, YT/Re)/(om + cn*kx)
            p_s = -A*cn*0.5*np.sqrt(cn*beta)*pp*np.sin(kx*XT)
        zeros = np.zeros((nz, ny, nx))
        super().set_initial_conditions(u[None], v[None], zeros, zeros, p_s)

In [ ]:
M = eq_waves1(par)
M.set_coriolis()
M.set_topography()
M.prepare_for_run()
M.set_initial_conditions()
h = M.halo
hov = []                            # p_s along the equator for the Hovmoeller diagram

In [ ]:
def plot(t, tau):
    clear_output(wait=True)
    p_s = M.p_s[tau, h:-h, h:-h]
    hov.append((t, p_s[ny//2]*1))
    plt.figure(figsize=(7, 6))
    plt.pcolormesh(M.xt_gl/1e3, M.yt_gl/1e3, p_s/9.81*100, cmap='RdBu_r', vmin=-A/9.81*100, vmax=A/9.81*100)
    plt.colorbar(label='$\\eta$ [cm]')
    s = 4
    plt.quiver(M.xt_gl[::s]/1e3, M.yt_gl[::s]/1e3, M.u[h, h:-h:s, h:-h:s], M.v[h, h:-h:s, h:-h:s])
    plt.xlabel('x [km]'); plt.ylabel('y [km]'); plt.title('%s wave, t = %.1f days' % (wave, t/86400)); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step_implicit_pressure(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt, tau)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
# Hovmoeller diagram along the equator
t = np.array([a[0] for a in hov]); p = np.array([a[1] for a in hov])
plt.figure(figsize=(6, 5))
plt.pcolormesh(M.xt_gl/1e3, t/86400, p/9.81*100, cmap='RdBu_r')
plt.colorbar(label='$\\eta$ [cm]'); plt.xlabel('x [km]'); plt.ylabel('t [days]')
plt.title('c_n = %.1f m/s' % cn); plt.show()